# 03 - Mixed Method, Attribution, and Impact

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [CVA package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: counterparty rows, netting-set rows, eligible hedge rows where applicable, and SA-CVA sensitivity rows for approved desks. The package dataset contract defines the fixture files and Arrow column-spec handoff: [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md). The staged usage path is described in the [CVA package journey](../docs/PACKAGE_JOURNEY.md).

This notebook combines an SA-CVA portfolio with a BA-CVA carve-out, then runs attribution and a finite-difference capital impact comparison.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
PACKAGE_ROOT = None
REPO_ROOT = None

for candidate in (HERE, *HERE.parents):
    if (candidate / "src" / "frtb_cva").exists():
        PACKAGE_ROOT = candidate
        REPO_ROOT = candidate.parent.parent if candidate.parent.name == "packages" else candidate
        break
    package_candidate = candidate / "packages" / "frtb-cva"
    if (package_candidate / "src" / "frtb_cva").exists():
        REPO_ROOT = candidate
        PACKAGE_ROOT = package_candidate
        break

if PACKAGE_ROOT is None or REPO_ROOT is None:
    raise RuntimeError("Could not locate the frtb-cva package root")

workspace_src_paths = tuple(sorted((REPO_ROOT / "packages").glob("*/src")))
for path in (
    PACKAGE_ROOT / "examples",
    PACKAGE_ROOT,
    PACKAGE_ROOT / "src",
    *workspace_src_paths,
    REPO_ROOT,
):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

try:
    from IPython.display import Markdown, display
except ImportError:

    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)


PACKAGE_ROOT

## Public API happy path

This notebook combines BA-CVA and SA-CVA branches through `calculate_cva_capital`, then uses the public attribution and impact helpers.


In [ ]:
from dataclasses import replace

from cva_notebook_data import markdown_table, sample_mixed_inputs
from frtb_cva import (
    assess_cva_capital_impact,
    attribute_cva_capital,
    calculate_cva_capital,
    validate_cva_result_reconciliation,
)

counterparties, netting_sets, sensitivities, context = sample_mixed_inputs()
baseline = calculate_cva_capital(context, counterparties, netting_sets, sensitivities=sensitivities)
validate_cva_result_reconciliation(baseline)

display(
    Markdown(
        markdown_table(
            ("metric", "value"),
            (
                ("method", baseline.method.value),
                ("total CVA capital", f"{baseline.total_cva_capital:,.2f}"),
                ("method components", len(baseline.method_components)),
            ),
        )
    )
)

## Implementation details / math verification - Component, attribution, and impact records

The remaining cells inspect mixed-method components, attribution rows, and finite-difference impact records.


In [ ]:
component_rows = tuple(
    (component.method.value, f"{component.total_capital:,.2f}", ", ".join(component.citations[:3]))
    for component in baseline.method_components
)

display(Markdown(markdown_table(("component", "capital", "citations"), component_rows)))

In [ ]:
attribution = attribute_cva_capital(baseline)
contribution_rows = tuple(
    (
        item.contribution_id,
        item.branch,
        f"{item.amount:,.2f}",
        item.method,
    )
    for item in attribution.contributions[:8]
)

display(
    Markdown(
        markdown_table(
            ("contribution", "branch", "amount", "method"),
            contribution_rows,
        )
    )
)
display(
    Markdown(
        markdown_table(
            ("attribution metric", "value"),
            (
                ("sum contributions", f"{attribution.sum_contributions:,.2f}"),
                ("residual", f"{attribution.residual:,.2f}"),
                ("unsupported branches", ", ".join(attribution.unsupported_branches)),
            ),
        )
    )
)

In [ ]:
candidate_sensitivities = (
    replace(sensitivities[0], amount=sensitivities[0].amount * 1.10),
    *sensitivities[1:],
)
candidate = calculate_cva_capital(
    replace(context, run_id="cva-notebook-mixed-candidate"),
    counterparties,
    netting_sets,
    sensitivities=candidate_sensitivities,
)
impact = assess_cva_capital_impact(baseline, candidate)

display(
    Markdown(
        markdown_table(
            ("baseline", "candidate", "delta", "method"),
            (
                (
                    f"{impact.baseline_total:,.2f}",
                    f"{impact.candidate_total:,.2f}",
                    f"{impact.delta:,.2f}",
                    impact.method,
                ),
            ),
        )
    )
)

## See also

- [CVA package journey](../docs/PACKAGE_JOURNEY.md)
- [CVA dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [IMA package journey](../../frtb-ima/docs/PACKAGE_JOURNEY.md)
